# Entrenamiento de Mask R-CNN: contorno de los espermatozoides
**Versión 0.4.0** (historial de cambios al final del cuaderno)

Este cuaderno entrena una red neuronal de **segmentación por instancias** (Mask R-CNN) que, dada una foto
de fluorescencia, dibuja el contorno de cada espermatozoide. Por ahora aprende **solo la forma**: el color no
se usa para decidir nada (indicación del cliente, chat del 16-oct-2026).

Está escrito para crecer: hoy se usa con 2 fotos, pero el código ya trabaja como si fueran 50 o más.

## Qué hace, paso a paso
1. **Prepara el entorno**: revisa que haya GPU y conecta Google Drive.
2. **Configuración**: todos los valores que se pueden cambiar están en una sola celda.
3. **Lee las etiquetas** de CVAT (formato COCO) y revisa que todo cuadre con las fotos.
4. **Separa entrenamiento y validación.** La validación son fotos (o zonas) que la red nunca ve al aprender;
   sirve para saber si de verdad funciona.
5. **Prepara las fotos** y las guarda en disco local para leerlas rápido.
6. **Arma los recortes de entrenamiento.** Las fotos son enormes y los espermatozoides diminutos, así que la
   red aprende con recortes de 1024 × 1024 px al tamaño real, sin reducir la foto.
7. **Construye la red** a partir de una Mask R-CNN ya preentrenada y la adapta a una sola clase.
8. **Entrena**, valida cada cierto tiempo, guarda el mejor modelo y puede **reanudarse** si Colab se corta.
9. **Revisa los resultados** con gráficas y con las fotos dibujadas.
10. **Exporta el modelo** para usarlo después en el servidor, sin GPU.

## Antes de empezar
- En Colab: *Entorno de ejecución → Cambiar tipo de entorno → GPU*.
- En tu Google Drive crea esta estructura:
  ```
  MyDrive/image-bio-pro/datos/imagenes/                 <- las fotos BMP etiquetadas
  MyDrive/image-bio-pro/datos/instances_default.json    <- el archivo COCO exportado de CVAT
  ```
- El archivo COCO está dentro del zip que exportaste de CVAT, en `annotations/instances_default.json`.
- Las fotos son del cliente: súbelas solo a tu Drive personal.
- `MODO_PRUEBA` viene en `False`: corre el entrenamiento real. Si quieres comprobar primero que todo funciona sin
  gastar GPU, ponlo en `True` (2 épocas muy cortas); guarda en su propia carpeta y no se mezcla con la corrida real.
- Si Colab se corta, vuelve a correr todo desde el principio: continúa donde se quedó.
- **Espacio en Drive:** cada vez que se reemplaza un archivo grande, Drive puede conservar la copia vieja en la
  papelera, y la papelera cuenta contra tu espacio. Por eso `ultimo.pt` (unos 350 MB) se guarda solo cada
  `GUARDAR_CADA` épocas. Vacía la papelera de Drive si se llena.

## Paso 1. Entorno
Colab ya trae PyTorch, torchvision, OpenCV y matplotlib; no hace falta instalar nada.

In [ ]:
import torch
import torchvision

print("PyTorch", torch.__version__, "| torchvision", torchvision.__version__)
print("GPU disponible:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("Tarjeta:", torch.cuda.get_device_name(0))

In [ ]:
# Conecta Google Drive (te pedirá permiso). Fuera de Colab se salta este paso.
try:
    from google.colab import drive
    drive.mount("/content/drive")
except ImportError:
    print("No estás en Colab: se usan las rutas tal como están en la configuración.")

## Paso 2. Configuración
Todo lo que se puede ajustar está aquí. Cada valor lleva su explicación.

In [ ]:
from pathlib import Path

VERSION_CUADERNO = "0.4.0"

# ---------- Rutas ----------
RUTA_BASE = Path("/content/drive/MyDrive/image-bio-pro")
CARPETA_IMAGENES = RUTA_BASE / "datos" / "imagenes"          # las fotos BMP
ARCHIVO_COCO = RUTA_BASE / "datos" / "instances_default.json"  # etiquetas exportadas de CVAT (COCO)
NOMBRE_CORRIDA = "corrida_01"                                  # cambia el nombre para no pisar corridas anteriores
CARPETA_CACHE = Path("/content/cache_imagenes")                # disco local de Colab (rápido, se borra al cerrar)
# La carpeta de salida (modelos, gráficas y revisiones) se calcula más abajo, después de revisar el modo.

# ---------- Modo ----------
MODO_PRUEBA = False       # True: entrena unos segundos solo para comprobar que todo funciona (en su propia carpeta)
PERMITIR_CPU = False      # False: si no hay GPU, se detiene (evita entrenar por error en CPU)
REANUDAR = True           # True: si ya existe ultimo.pt en la carpeta de salida, continúa desde ahí
INCLUIR_FOTOS_SIN_ETIQUETAS = False
# False: las fotos con cero etiquetas se omiten (lo normal es que sean fotos que aún no se etiquetan).
# True: se usan como ejemplos de "aquí no hay espermatozoides", por ejemplo fotos de tejido. Úsalo solo a propósito.

# ---------- Clase ----------
NOMBRE_CLASE = "cabeza"   # etiqueta de CVAT. Por ahora significa "espermatozoide visible completo"

# ---------- Preparación de la imagen ----------
USAR_GRISES = True
# True: la red ve solo el brillo (el valor más alto de rojo, verde y azul de cada píxel), no el color.
#       Así funciona igual con tinciones de distinto color (petición del cliente).
INVERTIR_SI_FONDO_CLARO = True
# True: si el fondo de la foto es claro (campo claro), se invierte el brillo: los espermatozoides oscuros sobre fondo
#       rosa quedan claros sobre fondo oscuro, como en fluorescencia. La idea es que la red aproveche lo que aprende de
#       un tipo de foto en el otro. Es una hipótesis sin comprobar: para medirla, entrena otra corrida con False y compara.

# ---------- Recortes y tamaño de la red ----------
LADO = 1024               # los recortes de entrenamiento y las teselas de inferencia miden LADO x LADO px
SOLAPE = 768              # traslape entre teselas al analizar una foto completa. Debe ser MAYOR que el objeto más grande
                          # (en campo claro el mayor mide 677 px; en fluorescencia, 116)
ESCALA_ENTRADA = 1.0      # 1.0 = la red ve los píxeles tal cual. 2.0 agranda al doble (más lento, ayuda con objetos muy chicos)
ANCLAS = (20, 48, 112, 256, 576)  # tamaños de caja con los que la red busca objetos, uno por nivel de la red.
                                  # Cubren de 10 px (fluorescencia) a unos 700 px (filamentos de campo claro)
ROI_MASCARA = 28          # detalle de la máscara de cada objeto: 28 da máscaras de 56 x 56 (torchvision usa 14 -> 28 x 28).
                          # Más detalle ayuda con filamentos largos y delgados; cuesta un poco más de memoria
MAX_DETECCIONES = 300     # máximo de objetos por tesela
UMBRAL_NMS_CAJAS = 0.6    # qué tanto pueden traslaparse dos cajas sin descartar una (los espermatozoides cruzados se traslapan)
UMBRAL_CONTENIDO = 0.5
# Limpieza de repetidos al final de detectar(): si la mitad o más de un contorno ya está dentro de otro con más seguridad,
# se descarta (la red a veces dibuja dos veces el mismo espermatozoide, sobre todo en los racimos).
# 0.5 salió mejor en una prueba con las dos fotos de entrenamiento (de 45 contornos de más bajó a 15). Más bajo quita más
# repetidos pero también espermatozoides buenos pegados a otros. None = sin limpieza.

# ---------- Separación entrenamiento / validación ----------
MIN_IMAGENES_SEPARAR_POR_IMAGEN = 10
# Con 10 fotos o más, algunas fotos completas se reservan para validar.
# Con menos, de CADA foto se reserva una franja (la parte más larga se corta).
FRACCION_IMAGENES_VALIDACION = 0.2   # con muchas fotos: qué parte se reserva
FRANJA_VALIDACION = 0.25             # con pocas fotos: qué fracción de cada foto se reserva
SEPARACION = 100                     # px de margen entre lo que se entrena y lo que se valida (para no mezclar un mismo espermatozoide).
                                     # Se deja en 100 para que las zonas de validación de fluorescencia sean las mismas que en
                                     # corrida_01 y corrida_02 (así se comparan). Los filamentos de campo claro miden más, así que
                                     # alguno puede quedar en las dos zonas; con tan pocas fotos de campo claro, su validación
                                     # aquí es poco confiable: pruébalo aparte con fotos que la red no haya visto.

# ---------- Entrenamiento ----------
EPOCAS = 60
RECORTES_POR_EPOCA = None  # None = automático: 8 por cada foto de entrenamiento, y nunca menos de 200 (cada recorte de
                           # 1024 px cubre lo que cuatro de 512)
TAMANO_LOTE = 2            # recortes por paso. Con recortes de 1024 px, 2 caben en la GPU gratuita de Colab
TRABAJADORES = 2           # procesos que preparan recortes en paralelo
LR = 0.0025                # tasa de aprendizaje; se ajusta al tamaño del lote (0.005 era para lotes de 4)
MOMENTUM = 0.9
DECAIMIENTO = 1e-4
ITER_CALENTAMIENTO = 100   # al principio la tasa de aprendizaje sube poco a poco
VALIDAR_CADA = 5           # cada cuántas épocas se valida
GUARDAR_CADA = 10          # cada cuántas épocas se guarda ultimo.pt (para poder reanudar). Pesa ~350 MB: no lo bajes mucho
                           # o llenarás el espacio de Drive

# ---------- Aumento de datos (variaciones al azar durante el entrenamiento) ----------
PROB_RECORTE_CON_CABEZA = 0.8   # el 80% de los recortes se centran en un espermatozoide; el resto son al azar (fibras, fondo)
ESCALA_AUG = (0.8, 1.25)        # se acerca o aleja la imagen un poco: sirve para fotos de distinto tamaño
MIN_VISIBLE = 0.3               # un espermatozoide cortado por el borde del recorte cuenta si se ve al menos esta fracción
MIN_PIXELES = 20                # y si le quedan al menos estos píxeles

# ---------- Resultados ----------
UMBRAL_DETECCION = 0.5     # seguridad mínima para contar un objeto durante el entrenamiento. Al final el cuaderno
                           # prueba varios umbrales y guarda en el modelo el que mejor funcione en la validación
SEMILLA = 42

if MODO_PRUEBA:
    EPOCAS, RECORTES_POR_EPOCA, VALIDAR_CADA, GUARDAR_CADA = 2, 16, 1, 1
    NOMBRE_CORRIDA += "_prueba"     # no se mezcla con una corrida real
    print("MODO_PRUEBA activo: 2 épocas muy cortas, solo para comprobar que todo corre.")

CARPETA_SALIDA = RUTA_BASE / "entrenamiento" / NOMBRE_CORRIDA   # modelos, gráficas y revisiones
print(f"Cuaderno versión {VERSION_CUADERNO} | corrida: {NOMBRE_CORRIDA}")

In [ ]:
import json
import math
import os
import time
import unicodedata

import cv2
import matplotlib.pyplot as plt
import numpy as np
from PIL import Image

Image.MAX_IMAGE_PIXELS = None   # las fotos del microscopio son enormes

dispositivo = torch.device("cuda" if torch.cuda.is_available() else "cpu")
if dispositivo.type == "cpu" and not PERMITIR_CPU:
    raise SystemExit("No hay GPU. En Colab: Entorno de ejecución > Cambiar tipo de entorno > GPU. "
                     "(o pon PERMITIR_CPU = True si sabes lo que haces)")

np.random.seed(SEMILLA)
torch.manual_seed(SEMILLA)
CARPETA_SALIDA.mkdir(parents=True, exist_ok=True)
print("Dispositivo:", dispositivo, "| salida:", CARPETA_SALIDA)

## Paso 3. Leer las etiquetas y revisar que todo cuadre
Se lee el archivo COCO de CVAT. Si falta una foto, si una foto no mide lo que dice el archivo, o si no existe la
etiqueta, aquí se detiene con un mensaje claro, antes de gastar tiempo de GPU.

In [ ]:
def _clave(nombre):
    """Nombre comparable: sin carpeta, con los acentos en una sola forma y sin importar mayúsculas."""
    return unicodedata.normalize("NFC", Path(nombre).name).casefold()


def _area_poligonos(poligonos):
    return float(sum(cv2.contourArea(p.astype(np.float32).reshape(-1, 1, 2)) for p in poligonos))


def leer_anotaciones():
    coco = json.loads(ARCHIVO_COCO.read_text(encoding="utf-8"))

    categorias = {c["name"]: c["id"] for c in coco["categories"]}
    if NOMBRE_CLASE not in categorias:
        raise ValueError(f"No existe la etiqueta '{NOMBRE_CLASE}'. Las que hay: {list(categorias)}")
    id_clase = categorias[NOMBRE_CLASE]

    archivos = {_clave(p.name): p for p in CARPETA_IMAGENES.iterdir() if p.is_file()}
    imagenes, faltan = {}, []
    for info in coco["images"]:
        ruta = archivos.get(_clave(info["file_name"]))
        if ruta is None:
            faltan.append(info["file_name"])
            continue
        with Image.open(ruta) as im:
            ancho, alto = im.size
        if (ancho, alto) != (info["width"], info["height"]):
            raise ValueError(f"{ruta.name} mide {ancho}x{alto} pero el archivo COCO dice "
                             f"{info['width']}x{info['height']}. ¿Es otra versión de la foto?")
        imagenes[info["id"]] = {"id": info["id"], "ruta": ruta, "nombre": ruta.name,
                                "ancho": ancho, "alto": alto, "instancias": []}
    if faltan:
        raise FileNotFoundError(f"Faltan estas fotos en {CARPETA_IMAGENES}:\n  " + "\n  ".join(faltan)
                                + f"\nFotos que sí hay: {sorted(p.name for p in CARPETA_IMAGENES.iterdir())[:10]}")

    omitidas = 0
    for a in coco["annotations"]:
        if a["category_id"] != id_clase or a["image_id"] not in imagenes:
            continue
        seg = a.get("segmentation")
        if not isinstance(seg, list) or a.get("iscrowd", 0):
            omitidas += 1               # máscaras en formato RLE: CVAT no las genera con polígonos
            continue
        polis = [np.array(s, np.float32).reshape(-1, 2) for s in seg if len(s) >= 6]
        if not polis:
            omitidas += 1
            continue
        todos = np.concatenate(polis)
        x0, y0 = todos.min(axis=0)
        x1, y1 = todos.max(axis=0)
        imagenes[a["image_id"]]["instancias"].append({
            "polis": polis,
            "caja": (float(x0), float(y0), float(x1), float(y1)),
            "centro": (float((x0 + x1) / 2), float((y0 + y1) / 2)),
            "area": _area_poligonos(polis),
        })
    if omitidas:
        print(f"Aviso: {omitidas} anotaciones omitidas (sin polígono válido).")
    todas = sorted(imagenes.values(), key=lambda i: i["id"])
    sin_etiquetas = [i for i in todas if not i["instancias"]]
    if sin_etiquetas and not INCLUIR_FOTOS_SIN_ETIQUETAS:
        # Una foto sin etiquetas casi siempre es una foto que todavía no se etiqueta; usarla enseñaría a la red que
        # ahí "no hay espermatozoides" y la validación contaría como error todo lo que encuentre.
        print(f"Aviso: se omiten {len(sin_etiquetas)} fotos sin ninguna etiqueta: "
              + ", ".join(i["nombre"] for i in sin_etiquetas[:5]) + ("..." if len(sin_etiquetas) > 5 else "")
              + "\n       (pon INCLUIR_FOTOS_SIN_ETIQUETAS = True si son ejemplos de 'no hay espermatozoides' a propósito)")
        todas = [i for i in todas if i["instancias"]]
    return todas


imagenes = leer_anotaciones()
total = sum(len(i["instancias"]) for i in imagenes)
if total == 0:
    raise ValueError("No hay ninguna anotación de la clase indicada.")

print(f"{len(imagenes)} fotos, {total} espermatozoides etiquetados\n")
print(f"{'foto':50s} {'tamaño':>12s} {'etiquetas':>10s}")
for im in imagenes:
    print(f"{im['nombre'][:50]:50s} {im['ancho']:>6d}x{im['alto']:<5d} {len(im['instancias']):>10d}")

lados = np.array([max(i["caja"][2] - i["caja"][0], i["caja"][3] - i["caja"][1])
                  for im in imagenes for i in im["instancias"]])
print(f"\nLado mayor de cada espermatozoide (px): mínimo {lados.min():.0f}, mediana {np.median(lados):.0f}, "
      f"máximo {lados.max():.0f}")
if lados.max() >= SOLAPE:
    print(f"AVISO: el espermatozoide más grande ({lados.max():.0f} px) no cabe en el traslape (SOLAPE = {SOLAPE}). "
          f"Sube SOLAPE o pueden duplicarse o cortarse en las orillas de las teselas.")

## Paso 4. Separar entrenamiento y validación
La **validación** son zonas que la red no ve al aprender. Con ellas se mide si funciona de verdad.

- **10 fotos o más:** se reservan algunas fotos completas.
- **Menos de 10 (como ahora, con 2):** de cada foto se reserva una franja del lado más largo, y se deja un margen
  para que un mismo espermatozoide no quede mitad en entrenamiento y mitad en validación.

In [ ]:
def crear_zonas(imagenes):
    """Cada zona es una región rectangular de una foto, marcada como entrenamiento o validación."""
    zonas = []
    if len(imagenes) >= MIN_IMAGENES_SEPARAR_POR_IMAGEN:
        rng = np.random.default_rng(SEMILLA)
        n_val = max(1, round(len(imagenes) * FRACCION_IMAGENES_VALIDACION))
        para_val = set(rng.permutation(len(imagenes))[:n_val].tolist())
        for i, im in enumerate(imagenes):
            zonas.append({"imagen": im, "uso": "val" if i in para_val else "train",
                          "caja": (0, 0, im["ancho"], im["alto"])})
    else:
        for im in imagenes:
            ancho, alto = im["ancho"], im["alto"]
            if ancho >= alto:
                corte = int(ancho * (1 - FRANJA_VALIDACION))
                entrena, valida = (0, 0, corte - SEPARACION, alto), (corte, 0, ancho, alto)
            else:
                corte = int(alto * (1 - FRANJA_VALIDACION))
                entrena, valida = (0, 0, ancho, corte - SEPARACION), (0, corte, ancho, alto)
            zonas.append({"imagen": im, "uso": "train", "caja": entrena})
            zonas.append({"imagen": im, "uso": "val", "caja": valida})
    return zonas


def _cuantas(zona):
    x0, y0, x1, y1 = zona["caja"]
    return sum(1 for i in zona["imagen"]["instancias"] if x0 <= i["centro"][0] < x1 and y0 <= i["centro"][1] < y1)


zonas = crear_zonas(imagenes)
print(f"{'foto':44s} {'uso':>6s} {'zona (x0,y0,x1,y1)':>28s} {'etiquetas':>10s}")
for z in zonas:
    print(f"{z['imagen']['nombre'][:44]:44s} {z['uso']:>6s} {str(z['caja']):>28s} {_cuantas(z):>10d}")
n_train = sum(_cuantas(z) for z in zonas if z["uso"] == "train")
n_val = sum(_cuantas(z) for z in zonas if z["uso"] == "val")
print(f"\nEntrenamiento: {n_train} espermatozoides | Validación: {n_val}")
if n_val < 20:
    print("AVISO: hay muy pocos espermatozoides para validar; las métricas serán poco confiables.")

if RECORTES_POR_EPOCA is None:
    fotos_entrenamiento = len({z["imagen"]["id"] for z in zonas if z["uso"] == "train"})
    RECORTES_POR_EPOCA = max(200, 8 * fotos_entrenamiento)
print(f"Recortes por época: {RECORTES_POR_EPOCA}")

## Paso 5. Preparar las fotos
Cada foto se lee **una sola vez**, se convierte (a brillo si `USAR_GRISES`) y se guarda como arreglo `.npy` en el
disco local de Colab. Después los recortes se sacan de ahí sin volver a decodificar el BMP (que pesa ~30 MB).

**Importante:** esta misma preparación se debe aplicar en el servidor al analizar una foto nueva.

In [ ]:
def preparar_imagen(rgb):
    """RGB (alto, ancho, 3) -> lo que ve la red. Debe ser idéntico en entrenamiento y en el servidor."""
    x = rgb.max(axis=2) if USAR_GRISES else rgb
    if INVERTIR_SI_FONDO_CLARO and np.median(x) > 127:
        x = 255 - x
    return np.ascontiguousarray(x)


def es_fondo_claro(ruta):
    """Campo claro (fondo claro) o fluorescencia (fondo oscuro), con el mismo criterio que preparar_imagen()."""
    with Image.open(ruta) as img:
        pequena = np.array(img.convert("RGB").reduce(8))
    return bool(np.median(pequena.max(axis=2)) > 127)


def preparar_cache(imagenes):
    CARPETA_CACHE.mkdir(parents=True, exist_ok=True)
    etiqueta = ("g" if USAR_GRISES else "c") + ("i" if INVERTIR_SI_FONDO_CLARO else "")
    for im in imagenes:
        destino = CARPETA_CACHE / f"{im['id']}_{etiqueta}.npy"
        if not destino.exists():
            with Image.open(im["ruta"]) as img:
                rgb = np.array(img.convert("RGB"))
            np.save(destino, preparar_imagen(rgb))
        im["cache"] = destino
        im["fondo_claro"] = es_fondo_claro(im["ruta"])


t0 = time.time()
preparar_cache(imagenes)
print(f"{len(imagenes)} fotos preparadas en {time.time() - t0:.0f} s: "
      f"{sum(not i['fondo_claro'] for i in imagenes)} de fluorescencia y {sum(i['fondo_claro'] for i in imagenes)} de campo claro")

## Paso 6. Recortes de entrenamiento
En cada paso la red recibe recortes de `LADO` × `LADO` px (1024). El **aumento de datos** agrega variaciones al azar para que
aprenda a reconocer la forma y no se aprenda las fotos de memoria:
volteos y giros de 90°, acercar o alejar un poco, más o menos brillo y contraste, y un poco de ruido.

In [ ]:
def mascara_local(inst, x0, y0, escala, ancho_max, alto_max):
    """Dibuja un espermatozoide dentro de una ventana que empieza en (x0, y0), con la escala indicada.
    Devuelve ((cx0, cy0, cx1, cy1), máscara) recortada a la ventana, o None si casi no se ve."""
    todos = [(p - (x0, y0)) * escala for p in inst["polis"]]
    minimo = np.min([p.min(axis=0) for p in todos], axis=0)
    maximo = np.max([p.max(axis=0) for p in todos], axis=0)
    cx0, cy0 = max(int(math.floor(minimo[0])), 0), max(int(math.floor(minimo[1])), 0)
    cx1, cy1 = min(int(math.ceil(maximo[0])) + 1, ancho_max), min(int(math.ceil(maximo[1])) + 1, alto_max)
    if cx1 <= cx0 or cy1 <= cy0:
        return None
    lienzo = np.zeros((cy1 - cy0, cx1 - cx0), np.uint8)
    cv2.fillPoly(lienzo, [np.round(p - (cx0, cy0)).astype(np.int32) for p in todos], 1)
    visibles = int(lienzo.sum())
    if visibles < max(MIN_PIXELES, MIN_VISIBLE * inst["area"] * escala ** 2):
        return None
    return (cx0, cy0, cx1, cy1), lienzo.astype(bool)


def agrupar(lote):
    return tuple(zip(*lote))


class RecortesEntrenamiento(torch.utils.data.Dataset):
    """Cada elemento es un recorte aleatorio (con su aumento) de las zonas de entrenamiento.
    El índice que recibe es una semilla: así cada recorte se puede repetir exactamente."""

    def __init__(self, zonas):
        self.zonas = [z for z in zonas if z["uso"] == "train"]
        if not self.zonas:
            raise ValueError("No hay zonas de entrenamiento.")
        areas = np.array([(z["caja"][2] - z["caja"][0]) * (z["caja"][3] - z["caja"][1]) for z in self.zonas], float)
        self.pesos_zona = areas / areas.sum()
        self.objetos = []                      # (zona, cx, cy) de cada espermatozoide de entrenamiento
        for zi, z in enumerate(self.zonas):
            x0, y0, x1, y1 = z["caja"]
            for inst in z["imagen"]["instancias"]:
                cx, cy = inst["centro"]
                if x0 <= cx < x1 and y0 <= cy < y1:
                    self.objetos.append((zi, cx, cy))

    def __len__(self):
        return RECORTES_POR_EPOCA

    def __getitem__(self, semilla):
        rng = np.random.default_rng((SEMILLA, int(semilla)))
        if self.objetos and rng.random() < PROB_RECORTE_CON_CABEZA:
            zi, cx, cy = self.objetos[rng.integers(len(self.objetos))]
        else:
            zi, cx, cy = int(rng.choice(len(self.zonas), p=self.pesos_zona)), None, None
        zona = self.zonas[zi]
        im = zona["imagen"]
        zx0, zy0, zx1, zy1 = zona["caja"]

        # Cuánto de la foto original cabe en un recorte (acercar o alejar un poco)
        factor = float(np.exp(rng.uniform(np.log(ESCALA_AUG[0]), np.log(ESCALA_AUG[1]))))
        lado_fuente = max(min(int(round(LADO / factor)), zx1 - zx0, zy1 - zy0), 8)
        escala = LADO / lado_fuente
        if cx is None:
            x0 = int(rng.integers(zx0, zx1 - lado_fuente + 1))
            y0 = int(rng.integers(zy0, zy1 - lado_fuente + 1))
        else:
            x0 = int(cx - lado_fuente / 2 + rng.uniform(-0.35, 0.35) * lado_fuente)
            y0 = int(cy - lado_fuente / 2 + rng.uniform(-0.35, 0.35) * lado_fuente)
            x0 = min(max(x0, zx0), zx1 - lado_fuente)
            y0 = min(max(y0, zy0), zy1 - lado_fuente)

        foto = np.load(im["cache"], mmap_mode="r")
        recorte = np.array(foto[y0:y0 + lado_fuente, x0:x0 + lado_fuente])
        if lado_fuente != LADO:
            interpolacion = cv2.INTER_AREA if lado_fuente > LADO else cv2.INTER_LINEAR
            recorte = cv2.resize(recorte, (LADO, LADO), interpolation=interpolacion)

        mascaras = []
        for inst in im["instancias"]:
            ix0, iy0, ix1, iy1 = inst["caja"]
            if ix1 < x0 or ix0 > x0 + lado_fuente or iy1 < y0 or iy0 > y0 + lado_fuente:
                continue
            resultado = mascara_local(inst, x0, y0, escala, LADO, LADO)
            if resultado is None:
                continue
            (a0, b0, a1, b1), m = resultado
            completa = np.zeros((LADO, LADO), np.uint8)
            completa[b0:b1, a0:a1] = m
            mascaras.append(completa)
        mascaras = np.stack(mascaras) if mascaras else np.zeros((0, LADO, LADO), np.uint8)

        # Volteos y giros
        if rng.random() < 0.5:
            recorte, mascaras = recorte[:, ::-1], mascaras[:, :, ::-1]
        if rng.random() < 0.5:
            recorte, mascaras = recorte[::-1], mascaras[:, ::-1]
        giros = int(rng.integers(4))
        recorte, mascaras = np.rot90(recorte, giros, axes=(0, 1)), np.rot90(mascaras, giros, axes=(1, 2))

        # Brillo, contraste y ruido
        x = recorte.astype(np.float32) * rng.uniform(0.7, 1.3) + rng.uniform(-20, 20)
        x += rng.normal(0, rng.uniform(0, 6), x.shape)
        if x.ndim == 3:
            x = x[..., rng.permutation(3)]          # si hay color, que no importe cuál canal es cuál
        x = np.clip(x, 0, 255) / 255.0
        imagen = np.repeat(x[None], 3, axis=0) if x.ndim == 2 else x.transpose(2, 0, 1)

        cajas, conservar = [], []
        for i, m in enumerate(mascaras):
            filas, cols = np.flatnonzero(m.any(1)), np.flatnonzero(m.any(0))
            if len(filas) == 0:
                continue
            cajas.append([cols[0], filas[0], cols[-1] + 1, filas[-1] + 1])
            conservar.append(i)
        mascaras = np.ascontiguousarray(mascaras[conservar] if conservar else mascaras[:0])
        objetivo = {
            "boxes": torch.as_tensor(np.array(cajas, np.float32).reshape(-1, 4)),
            "labels": torch.ones(len(cajas), dtype=torch.int64),
            "masks": torch.from_numpy(mascaras),
        }
        return torch.from_numpy(np.ascontiguousarray(imagen, dtype=np.float32)), objetivo


datos_entrenamiento = RecortesEntrenamiento(zonas)
print(f"{len(datos_entrenamiento.objetos)} espermatozoides disponibles para centrar recortes")

### Revisión visual de los recortes
Mira estos recortes antes de entrenar: el contorno verde debe quedar sobre cada espermatozoide, incluso girado
o con otro brillo. Si algo se ve desfasado, hay un error en los datos y conviene detenerse aquí.

In [ ]:
def contornos_sobre(imagen_chw, mascaras):
    vista = (imagen_chw.numpy().transpose(1, 2, 0) * 255).astype(np.uint8).copy()
    for m in mascaras.numpy():
        cs, _ = cv2.findContours(m.astype(np.uint8), cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
        cv2.drawContours(vista, cs, -1, (0, 255, 0), 1)
    return vista


fig, ejes = plt.subplots(2, 3, figsize=(15, 10))
for eje, semilla in zip(ejes.ravel(), range(1000, 1006)):
    img, obj = datos_entrenamiento[semilla]
    eje.imshow(contornos_sobre(img, obj["masks"]))
    eje.set_title(f"{len(obj['boxes'])} espermatozoides")
    eje.axis("off")
plt.tight_layout()
plt.show()

## Paso 7. La red
Se parte de una **Mask R-CNN preentrenada con COCO** (millones de objetos cotidianos). Ya sabe ver bordes y formas;
aquí solo se le enseña que existe **una clase**: `cabeza`. Esto se llama *transfer learning* y funciona con pocas fotos.

Ajustes pensados para este problema: anclas desde 20 hasta 576 px (los objetos van de 10 px en fluorescencia a casi
700 px en campo claro), máscaras con más detalle para los filamentos delgados (`ROI_MASCARA`), la entrada no se
reescala (`min_size = max_size = LADO`) para no perder detalle, y hasta 300 detecciones por tesela.

Se usa la versión con capas de normalización congeladas (`maskrcnn_resnet50_fpn`) porque es estable con lotes
pequeños, como los que caben en la GPU gratuita de Colab.

In [ ]:
from torchvision.models.detection import MaskRCNN_ResNet50_FPN_Weights, maskrcnn_resnet50_fpn
from torchvision.models.detection.anchor_utils import AnchorGenerator
from torchvision.models.detection.faster_rcnn import FastRCNNPredictor
from torchvision.models.detection.mask_rcnn import MaskRCNNPredictor
from torchvision.ops import MultiScaleRoIAlign
from torchvision.ops.misc import FrozenBatchNorm2d


def _congelar_normalizacion(modulo, eps):
    """Cambia cada capa BatchNorm2d por una FrozenBatchNorm2d (como en el entrenamiento), conservando su lugar."""
    for nombre, hijo in modulo.named_children():
        if isinstance(hijo, torch.nn.BatchNorm2d):
            setattr(modulo, nombre, FrozenBatchNorm2d(hijo.num_features, eps=eps))
        else:
            _congelar_normalizacion(hijo, eps)


def construir_modelo(preentrenado, bn_eps=None):
    """preentrenado=True: para entrenar (descarga los pesos de COCO). preentrenado=False: para cargar un modelo ya
    entrenado, sin internet; ahí hay que pasar el bn_eps guardado dentro del modelo (ver más abajo)."""
    generador = AnchorGenerator(sizes=tuple((s,) for s in ANCLAS),
                                aspect_ratios=((0.5, 1.0, 2.0),) * len(ANCLAS))
    lado = int(LADO * ESCALA_ENTRADA)
    opciones = dict(min_size=lado, max_size=lado, rpn_anchor_generator=generador,
                    box_detections_per_img=MAX_DETECCIONES, box_nms_thresh=UMBRAL_NMS_CAJAS,
                    mask_roi_pool=MultiScaleRoIAlign(featmap_names=["0", "1", "2", "3"], output_size=ROI_MASCARA,
                                                     sampling_ratio=2))
    if preentrenado:
        modelo = maskrcnn_resnet50_fpn(weights=MaskRCNN_ResNet50_FPN_Weights.COCO_V1,
                                       trainable_backbone_layers=3, **opciones)
    else:
        modelo = maskrcnn_resnet50_fpn(weights=None, weights_backbone=None, **opciones)
    # Dos clases: fondo y espermatozoide
    entradas = modelo.roi_heads.box_predictor.cls_score.in_features
    modelo.roi_heads.box_predictor = FastRCNNPredictor(entradas, 2)
    canales = modelo.roi_heads.mask_predictor.conv5_mask.in_channels
    modelo.roi_heads.mask_predictor = MaskRCNNPredictor(canales, 256, 2)
    if bn_eps is not None:
        # Al entrenar, torchvision usa capas de normalización CONGELADAS con su propio eps (puede ser 0.0). Al cargar sin
        # los pesos de COCO se arman capas normales con eps = 1e-5, y esa diferencia mínima se acumula hasta cambiar los
        # resultados. Por eso se cambian por las mismas capas congeladas y con el mismo eps del entrenamiento
        # (no sirve poner eps = 0 en una capa normal: PyTorch lo rechaza).
        _congelar_normalizacion(modelo, bn_eps)
    return modelo


modelo = construir_modelo(preentrenado=True).to(dispositivo)
n_param = sum(p.numel() for p in modelo.parameters())
n_entrenables = sum(p.numel() for p in modelo.parameters() if p.requires_grad)
print(f"Parámetros: {n_param / 1e6:.1f} M ({n_entrenables / 1e6:.1f} M se ajustan al entrenar)")

## Paso 8. Cómo se mide si funciona
Para validar se analiza cada zona de validación completa, **igual que en producción**: se corta en teselas con
traslape, se analiza cada una, se juntan los resultados quitando duplicados entre teselas y, al final, se quitan los
contornos repetidos dentro de un mismo espermatozoide (`UMBRAL_CONTENIDO`).

Métricas (un espermatozoide detectado cuenta como acierto si su contorno coincide al menos 50% con el etiquetado):
- **Precisión**: de lo que la red marcó, qué parte era un espermatozoide de verdad.
- **Exhaustividad (recall)**: de los espermatozoides etiquetados, qué parte encontró.
- **F1**: promedio de las dos.
- **AP50**: calidad global sin elegir un umbral de seguridad.

In [ ]:
def interseccion_mascaras(a, b):
    """Cuántos píxeles comparten dos contornos. Cada uno trae su caja y su máscara local."""
    ax0, ay0, ax1, ay1 = a["caja"]
    bx0, by0, bx1, by1 = b["caja"]
    x0, y0, x1, y1 = max(ax0, bx0), max(ay0, by0), min(ax1, bx1), min(ay1, by1)
    if x1 <= x0 or y1 <= y0:
        return 0
    ma = a["mascara"][y0 - ay0:y1 - ay0, x0 - ax0:x1 - ax0]
    mb = b["mascara"][y0 - by0:y1 - by0, x0 - bx0:x1 - bx0]
    return int(np.count_nonzero(ma & mb))


def iou_mascaras(a, b):
    """Qué tanto se traslapan dos contornos (0 a 1)."""
    inter = interseccion_mascaras(a, b)
    if inter == 0:
        return 0.0
    return inter / (a["area"] + b["area"] - inter)


def quitar_repetidos(detecciones, umbral_contenido):
    """De mayor a menor seguridad, descarta un contorno si al menos `umbral_contenido` de su área ya está
    cubierta por uno que se conservó. Quita los espermatozoides dibujados dos veces o en pedazos."""
    conservadas = []
    for d in sorted(detecciones, key=lambda d: -d["puntaje"]):
        if all(interseccion_mascaras(d, k) / d["area"] < umbral_contenido for k in conservadas):
            conservadas.append(d)
    return conservadas


def _posiciones(n, lado, paso):
    if n <= lado:
        return [0]
    return list(range(0, n - lado, paso)) + [n - lado]


@torch.no_grad()
def detectar(modelo, arreglo, dispositivo, umbral_minimo=0.05, lote=4):
    """Analiza una foto (o zona) ya preparada, por teselas con traslape.
    Devuelve una lista de {caja, mascara, area, puntaje} en coordenadas de la foto."""
    modelo.eval()
    alto, ancho = arreglo.shape[:2]
    paso = LADO - SOLAPE
    teselas = [(x, y) for y in _posiciones(alto, LADO, paso) for x in _posiciones(ancho, LADO, paso)]
    encontrados = []
    for i in range(0, len(teselas), lote):
        grupo = teselas[i:i + lote]
        entradas = []
        for x, y in grupo:
            t = np.asarray(arreglo[y:y + LADO, x:x + LADO])
            if t.shape[0] < LADO or t.shape[1] < LADO:
                relleno = [(0, LADO - t.shape[0]), (0, LADO - t.shape[1])] + [(0, 0)] * (t.ndim - 2)
                t = np.pad(t, relleno)
            t = np.repeat(t[None], 3, axis=0) if t.ndim == 2 else t.transpose(2, 0, 1)
            entradas.append(torch.from_numpy(np.ascontiguousarray(t)).float().div(255).to(dispositivo))
        with torch.autocast(device_type=dispositivo.type, enabled=dispositivo.type == "cuda"):
            salidas = modelo(entradas)
        for (x, y), salida in zip(grupo, salidas):
            mascaras = (salida["masks"][:, 0] > 0.5).cpu().numpy()
            puntajes = salida["scores"].float().cpu().numpy()
            for m, puntaje in zip(mascaras, puntajes):
                if puntaje < umbral_minimo:
                    continue
                filas, cols = np.flatnonzero(m.any(1)), np.flatnonzero(m.any(0))
                if len(filas) == 0:
                    continue
                f0, f1, c0, c1 = filas[0], filas[-1] + 1, cols[0], cols[-1] + 1
                # Si toca un borde interno de la tesela, el objeto puede estar cortado: lo verá entero la tesela vecina
                if ((c0 <= 0 and x > 0) or (c1 >= LADO and x + LADO < ancho)
                        or (f0 <= 0 and y > 0) or (f1 >= LADO and y + LADO < alto)):
                    continue
                local = m[f0:f1, c0:c1][:max(alto - (y + f0), 0), :max(ancho - (x + c0), 0)]
                if local.size == 0 or not local.any():
                    continue
                encontrados.append({"caja": (x + c0, y + f0, x + c0 + local.shape[1], y + f0 + local.shape[0]),
                                    "mascara": local, "area": int(local.sum()), "puntaje": float(puntaje)})
    # Duplicados entre teselas: se queda el de mayor seguridad
    conservados = []
    for d in sorted(encontrados, key=lambda d: -d["puntaje"]):
        if all(iou_mascaras(d, k) < 0.5 for k in conservados):
            conservados.append(d)
    # Repetidos dentro de un mismo espermatozoide (la red a veces lo dibuja dos veces o en pedazos)
    if UMBRAL_CONTENIDO:
        conservados = quitar_repetidos(conservados, UMBRAL_CONTENIDO)
    return conservados


def etiquetas_de_zona(zona):
    """Los espermatozoides etiquetados dentro de una zona, en coordenadas de la zona."""
    x0, y0, x1, y1 = zona["caja"]
    resultado = []
    for inst in zona["imagen"]["instancias"]:
        ix0, iy0, ix1, iy1 = inst["caja"]
        if ix1 < x0 or ix0 > x1 or iy1 < y0 or iy0 > y1:
            continue
        r = mascara_local(inst, x0, y0, 1.0, x1 - x0, y1 - y0)
        if r is not None:
            caja, m = r
            resultado.append({"caja": caja, "mascara": m, "area": int(m.sum())})
    return resultado


def aciertos_de_zona(detecciones, etiquetadas, umbral_iou=0.5):
    """Empareja detecciones con etiquetas (de la más segura a la menos). Devuelve [(puntaje, acierto)]."""
    usadas = [False] * len(etiquetadas)
    pares = []
    for d in sorted(detecciones, key=lambda d: -d["puntaje"]):
        mejor, indice = 0.0, -1
        for j, e in enumerate(etiquetadas):
            if usadas[j]:
                continue
            v = iou_mascaras(d, e)
            if v > mejor:
                mejor, indice = v, j
        if mejor >= umbral_iou:
            usadas[indice] = True
        pares.append((d["puntaje"], mejor >= umbral_iou))
    return pares


def metricas(pares, n_etiquetadas, umbral=UMBRAL_DETECCION):
    pares = sorted(pares, key=lambda p: -p[0])
    aciertos = np.array([p[1] for p in pares], dtype=float)
    acum_tp, acum_fp = np.cumsum(aciertos), np.cumsum(1 - aciertos)
    recall = acum_tp / max(n_etiquetadas, 1)
    precision = acum_tp / np.maximum(acum_tp + acum_fp, 1e-9)
    mrec = np.concatenate(([0.0], recall, [1.0]))
    mpre = np.concatenate(([0.0], precision, [0.0]))
    for i in range(len(mpre) - 2, -1, -1):
        mpre[i] = max(mpre[i], mpre[i + 1])
    cambios = np.where(mrec[1:] != mrec[:-1])[0]
    ap50 = float(np.sum((mrec[cambios + 1] - mrec[cambios]) * mpre[cambios + 1]))
    k = sum(1 for p in pares if p[0] >= umbral)
    tp = int(aciertos[:k].sum())
    prec = tp / k if k else 0.0
    rec = tp / n_etiquetadas if n_etiquetadas else 0.0
    f1 = 2 * prec * rec / (prec + rec) if prec + rec else 0.0
    return {"ap50": round(ap50, 4), "precision": round(prec, 4), "recall": round(rec, 4), "f1": round(f1, 4),
            "detectadas": k, "etiquetadas": n_etiquetadas}


def pares_de_validacion(modelo, filtro=None):
    """Analiza las zonas de validación y devuelve [(seguridad, acierto)] y cuántos espermatozoides hay etiquetados.
    filtro(imagen) -> bool permite medir solo un tipo de foto."""
    pares, total_etiquetadas = [], 0
    for z in (z for z in zonas if z["uso"] == "val" and (filtro is None or filtro(z["imagen"]))):
        x0, y0, x1, y1 = z["caja"]
        foto = np.load(z["imagen"]["cache"], mmap_mode="r")
        detecciones = detectar(modelo, foto[y0:y1, x0:x1], dispositivo)
        etiquetadas = etiquetas_de_zona(z)
        pares += aciertos_de_zona(detecciones, etiquetadas)
        total_etiquetadas += len(etiquetadas)
    return pares, total_etiquetadas


def validar(modelo):
    return metricas(*pares_de_validacion(modelo))

## Paso 9. Entrenar
Cada época toma `RECORTES_POR_EPOCA` recortes nuevos al azar. Detalles:
- **SGD con calentamiento y descenso suave** de la tasa de aprendizaje.
- **Precisión mixta** (AMP) en GPU: más rápido y con menos memoria.
- Cada `VALIDAR_CADA` épocas se valida y, si mejora el AP50, se guarda `mejor.pt`.
- Cada `GUARDAR_CADA` épocas se guarda `ultimo.pt` (con el optimizador), y **si Colab se corta basta con volver a correr
  todo desde el principio**: continúa desde la última época guardada.

In [ ]:
def escribir_atomico(objeto, ruta):
    """Guarda en un archivo temporal y luego lo renombra: un corte a medias no deja un archivo roto."""
    temporal = ruta.with_suffix(ruta.suffix + ".tmp")
    torch.save(objeto, temporal)
    os.replace(temporal, ruta)


def configuracion_del_modelo():
    """Todo lo que el servidor necesita saber para usar el modelo igual que aquí."""
    return {"lado": LADO, "solape": SOLAPE, "escala_entrada": ESCALA_ENTRADA, "anclas": list(ANCLAS),
            "max_detecciones": MAX_DETECCIONES, "umbral_nms_cajas": UMBRAL_NMS_CAJAS,
            "umbral_contenido": UMBRAL_CONTENIDO, "roi_mascara": ROI_MASCARA,
            "usar_grises": USAR_GRISES, "invertir_si_fondo_claro": INVERTIR_SI_FONDO_CLARO,
            "umbral_deteccion": UMBRAL_DETECCION, "clase": NOMBRE_CLASE,
            "version_cuaderno": VERSION_CUADERNO, "bn_eps": float(modelo.backbone.body.bn1.eps),
            "torch": torch.__version__, "torchvision": torchvision.__version__}


def configuracion_de_entrenamiento():
    """Lo que no se puede cambiar a la mitad de una corrida: el modelo y cómo se entrena."""
    base = configuracion_del_modelo()
    for clave in ("umbral_deteccion", "umbral_contenido", "torch", "torchvision", "version_cuaderno"):
        base.pop(clave)     # esto sí puede cambiar entre una sesión y otra sin que sea otro entrenamiento
    base.update({"epocas": EPOCAS, "recortes_por_epoca": RECORTES_POR_EPOCA, "tamano_lote": TAMANO_LOTE,
                 "lr": LR, "momentum": MOMENTUM, "decaimiento": DECAIMIENTO, "calentamiento": ITER_CALENTAMIENTO,
                 "escala_aug": list(ESCALA_AUG), "semilla": SEMILLA, "fotos": [i["nombre"] for i in imagenes]})
    return base


parametros = [p for p in modelo.parameters() if p.requires_grad]
optimizador = torch.optim.SGD(parametros, lr=LR, momentum=MOMENTUM, weight_decay=DECAIMIENTO)
iter_por_epoca = math.ceil(RECORTES_POR_EPOCA / TAMANO_LOTE)
iter_totales = EPOCAS * iter_por_epoca


def factor_lr(it):
    if it < ITER_CALENTAMIENTO:
        return (it + 1) / ITER_CALENTAMIENTO
    avance = (it - ITER_CALENTAMIENTO) / max(iter_totales - ITER_CALENTAMIENTO, 1)
    return 0.01 + 0.99 * 0.5 * (1 + math.cos(math.pi * min(avance, 1.0)))


planificador = torch.optim.lr_scheduler.LambdaLR(optimizador, factor_lr)
usar_amp = dispositivo.type == "cuda"
escalador = torch.amp.GradScaler("cuda", enabled=usar_amp)

historial, mejor_ap, epoca_inicial = [], -1.0, 0
ruta_ultimo = CARPETA_SALIDA / "ultimo.pt"
if REANUDAR and ruta_ultimo.exists():
    punto = torch.load(ruta_ultimo, map_location=dispositivo, weights_only=False)
    # Reanudar con otra configuración mezclaría dos entrenamientos distintos: mejor detenerse
    guardada, actual = punto.get("configuracion", {}), configuracion_de_entrenamiento()
    distintas = {k: (guardada.get(k), actual[k]) for k in actual if guardada.get(k) != actual[k]}
    if distintas:
        raise RuntimeError(
            f"Ya hay una corrida guardada en {CARPETA_SALIDA} con otra configuración:\n"
            + "\n".join(f"  {k}: guardada {v[0]!r}, ahora {v[1]!r}" for k, v in distintas.items())
            + "\nCambia NOMBRE_CORRIDA para empezar una corrida nueva, o REANUDAR = False para empezar de cero aquí "
              "(sobrescribe lo anterior), o regresa la configuración a como estaba.")
    modelo.load_state_dict(punto["modelo"])
    optimizador.load_state_dict(punto["optimizador"])
    planificador.load_state_dict(punto["planificador"])
    escalador.load_state_dict(punto["escalador"])
    historial, mejor_ap, epoca_inicial = punto["historial"], punto["mejor_ap"], punto["epoca"] + 1
    if epoca_inicial >= EPOCAS:
        print(f"Esta corrida ya terminó ({EPOCAS} épocas): se omite el entrenamiento y se pasa a los resultados.")
    else:
        print(f"Reanudando desde la época {epoca_inicial + 1} de {EPOCAS}")

In [ ]:
t_inicio = time.time()
for epoca in range(epoca_inicial, EPOCAS):
    modelo.train()
    semillas = list(range(epoca * RECORTES_POR_EPOCA, (epoca + 1) * RECORTES_POR_EPOCA))
    cargador = torch.utils.data.DataLoader(datos_entrenamiento, batch_size=TAMANO_LOTE, sampler=semillas,
                                           num_workers=TRABAJADORES, collate_fn=agrupar)
    perdidas = []
    for imagenes_lote, objetivos in cargador:
        imagenes_lote = [i.to(dispositivo) for i in imagenes_lote]
        objetivos = [{k: v.to(dispositivo) for k, v in o.items()} for o in objetivos]
        with torch.autocast(device_type=dispositivo.type, enabled=usar_amp):
            componentes = modelo(imagenes_lote, objetivos)
            perdida = sum(componentes.values())
        if not torch.isfinite(perdida):
            raise FloatingPointError(f"La pérdida se volvió {perdida.item()} en la época {epoca + 1}. "
                                     f"Baja LR e inténtalo de nuevo. Componentes: "
                                     f"{ {k: float(v) for k, v in componentes.items()} }")
        optimizador.zero_grad(set_to_none=True)
        escalador.scale(perdida).backward()
        escalador.unscale_(optimizador)
        torch.nn.utils.clip_grad_norm_(parametros, 10.0)
        escalador.step(optimizador)
        escalador.update()
        planificador.step()
        perdidas.append(float(perdida.detach()))

    registro = {"epoca": epoca + 1, "perdida": round(float(np.mean(perdidas)), 4),
                "lr": round(optimizador.param_groups[0]["lr"], 6)}
    if (epoca + 1) % VALIDAR_CADA == 0 or epoca + 1 == EPOCAS:
        registro["validacion"] = validar(modelo)
        ap = registro["validacion"]["ap50"]
        if ap > mejor_ap:
            mejor_ap = ap
            escribir_atomico({"state_dict": {k: v.cpu() for k, v in modelo.state_dict().items()},
                              "config": configuracion_del_modelo(), "epoca": epoca + 1, "ap50": ap},
                             CARPETA_SALIDA / "mejor.pt")
            registro["mejor"] = True
    historial.append(registro)
    if (epoca + 1) % GUARDAR_CADA == 0 or epoca + 1 == EPOCAS:
        escribir_atomico({"modelo": modelo.state_dict(), "optimizador": optimizador.state_dict(),
                          "planificador": planificador.state_dict(), "escalador": escalador.state_dict(),
                          "historial": historial, "mejor_ap": mejor_ap, "epoca": epoca,
                          "configuracion": configuracion_de_entrenamiento()}, ruta_ultimo)
    (CARPETA_SALIDA / "historial.json").write_text(json.dumps(historial, indent=1), encoding="utf-8")
    minutos = (time.time() - t_inicio) / 60
    linea = f"Época {epoca + 1:3d}/{EPOCAS}  pérdida {registro['perdida']:.4f}  ({minutos:.1f} min)"
    if "validacion" in registro:
        v = registro["validacion"]
        linea += (f"  | validación: AP50 {v['ap50']:.3f}  precisión {v['precision']:.3f}  "
                  f"recall {v['recall']:.3f}  F1 {v['f1']:.3f}" + ("  <- mejor" if registro.get("mejor") else ""))
    print(linea)

## Paso 10. Resultados
**Cómo leer las gráficas:**
- La **pérdida** debe bajar. Si sube o se queda plana desde el inicio, algo anda mal.
- Las **métricas de validación** deben subir. Si la pérdida baja pero la validación no mejora (o empeora), la red se
  está aprendiendo las fotos de memoria (*sobreajuste*): hacen falta más fotos etiquetadas.

In [ ]:
epocas = [h["epoca"] for h in historial]
validadas = [h for h in historial if "validacion" in h]
fig, ejes = plt.subplots(1, 2, figsize=(13, 4))
ejes[0].plot(epocas, [h["perdida"] for h in historial])
ejes[0].set(title="Pérdida de entrenamiento", xlabel="época")
if validadas:
    for clave in ("ap50", "precision", "recall", "f1"):
        ejes[1].plot([h["epoca"] for h in validadas], [h["validacion"][clave] for h in validadas], marker="o", label=clave)
    ejes[1].set(title="Validación", xlabel="época", ylim=(0, 1))
    ejes[1].legend()
plt.tight_layout()
plt.savefig(CARPETA_SALIDA / "curvas.png", dpi=120)
plt.show()

In [ ]:
# Se carga el mejor modelo (no necesariamente el de la última época) y se mide una última vez
punto = torch.load(CARPETA_SALIDA / "mejor.pt", map_location=dispositivo, weights_only=False)
modelo.load_state_dict(punto["state_dict"])
print(f"Mejor modelo: época {punto['epoca']}, AP50 de validación {punto['ap50']:.3f}")
pares_val, n_val_etiquetadas = pares_de_validacion(modelo)
final = metricas(pares_val, n_val_etiquetadas)
print("Validación final (umbral 0.5):", final)

# ¿Con qué seguridad mínima conviene contar un objeto? Se prueban varios umbrales sobre la validación.
# Subirlo evita falsos positivos (fibras); bajarlo encuentra más espermatozoides.
barrido = {u: metricas(pares_val, n_val_etiquetadas, umbral=u) for u in (0.3, 0.4, 0.5, 0.6, 0.7, 0.8, 0.9)}
print(f"\n{'umbral':>7s} {'precisión':>10s} {'recall':>8s} {'F1':>6s} {'detectadas':>11s}")
for u, m in barrido.items():
    print(f"{u:7.1f} {m['precision']:10.3f} {m['recall']:8.3f} {m['f1']:6.3f} {m['detectadas']:11d}")
umbral_final = max(barrido, key=lambda u: (barrido[u]["f1"], -abs(u - 0.5)))   # mejor F1; en empate, el más cercano a 0.5
print(f"\nUmbral elegido: {umbral_final} (mejor F1 en la validación)")
if n_val_etiquetadas < 50:
    print(f"AVISO: la validación tiene solo {n_val_etiquetadas} espermatozoides; este umbral es una estimación poco firme. "
          f"Vuelve a elegirlo cuando haya más fotos.")
# Por tipo de foto: así se ve si sumar campo claro empeoró la fluorescencia (compárala con la corrida anterior)
por_tipo = {}
print("\nCon el umbral elegido, por tipo de foto (zonas de validación):")
for tipo, filtro in (("fluorescencia", lambda im: not im["fondo_claro"]), ("campo claro", lambda im: im["fondo_claro"])):
    p_tipo, n_tipo = pares_de_validacion(modelo, filtro)
    if n_tipo or p_tipo:
        por_tipo[tipo] = metricas(p_tipo, n_tipo, umbral=umbral_final)
        m = por_tipo[tipo]
        print(f"  {tipo:14s} etiquetados {m['etiquetadas']:3d} | detectados {m['detectadas']:3d} | precisión {m['precision']:.3f} "
              f"| recall {m['recall']:.3f} | F1 {m['f1']:.3f}")
        if n_tipo < 20:
            print(f"  {'':14s} (con menos de 20 espermatozoides esta cifra es muy inestable)")
(CARPETA_SALIDA / "metricas_finales.json").write_text(json.dumps(
    {"umbral_elegido": umbral_final, "barrido": {str(u): m for u, m in barrido.items()}, "con_0.5": final,
     "por_tipo": por_tipo}, indent=1), encoding="utf-8")

### Las fotos con lo que encontró la red
Verde = lo que detectó la red. Blanco = lo que etiquetaste. El rectángulo amarillo marca la zona de **validación**:
solo ahí cuenta como prueba real; fuera de ella la red ya vio esos espermatozoides al entrenar.

In [ ]:
CARPETA_REVISION = CARPETA_SALIDA / "revision"
CARPETA_REVISION.mkdir(exist_ok=True)
filas_csv = ["foto,etiquetados,detectados"]
for im in imagenes:
    foto = np.load(im["cache"], mmap_mode="r")
    detecciones = [d for d in detectar(modelo, foto, dispositivo) if d["puntaje"] >= umbral_final]
    vista = np.array(foto) if foto.ndim == 3 else np.repeat(np.array(foto)[..., None], 3, axis=2)
    vista = np.ascontiguousarray(vista)
    for inst in im["instancias"]:
        cv2.polylines(vista, [np.round(p).astype(np.int32) for p in inst["polis"]], True, (255, 255, 255), 1)
    for d in detecciones:
        x0, y0, _, _ = d["caja"]
        cs, _ = cv2.findContours(d["mascara"].astype(np.uint8), cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
        cv2.drawContours(vista, cs, -1, (0, 255, 0), 1, offset=(x0, y0))
    for z in zonas:
        if z["imagen"] is im and z["uso"] == "val":
            cv2.rectangle(vista, z["caja"][:2], z["caja"][2:], (255, 255, 0), 3)
    reduccion = 1600 / max(vista.shape[:2])
    if reduccion < 1:
        vista = cv2.resize(vista, None, fx=reduccion, fy=reduccion, interpolation=cv2.INTER_AREA)
    cv2.imwrite(str(CARPETA_REVISION / f"foto_{im['id']}.png"), cv2.cvtColor(vista, cv2.COLOR_RGB2BGR))
    filas_csv.append(f"\"{im['nombre']}\",{len(im['instancias'])},{len(detecciones)}")
    print(f"{im['nombre']}: {len(im['instancias'])} etiquetados, {len(detecciones)} detectados")
    plt.figure(figsize=(12, 9))
    plt.imshow(vista)
    plt.axis("off")
    plt.title(im["nombre"])
    plt.show()
(CARPETA_SALIDA / "conteos.csv").write_text("\n".join(filas_csv), encoding="utf-8")

## Paso 11. Exportar el modelo
Se guarda un solo archivo, `modelo_cabezas.pt`, con los pesos **y** la configuración con la que se entrenó
(tamaño de tesela, traslape, si usa grises, umbral...). El servidor lo lee en CPU, sin GPU.

Enseguida se comprueba que el archivo se puede cargar en CPU desde cero y que da lo mismo que el modelo en memoria.
Esa misma celda mide cuánto tarda una tesela en CPU, para estimar el tiempo de análisis de una foto en el servidor.

In [ ]:
ruta_modelo = CARPETA_SALIDA / "modelo_cabezas.pt"
config_exportada = dict(configuracion_del_modelo(), umbral_deteccion=umbral_final)   # el umbral que salió mejor
torch.save({"version": VERSION_CUADERNO, "state_dict": {k: v.cpu() for k, v in modelo.state_dict().items()},
            "config": config_exportada, "epoca": punto["epoca"], "ap50_validacion": punto["ap50"]},
           ruta_modelo)
print(f"Guardado: {ruta_modelo} ({ruta_modelo.stat().st_size / 1e6:.0f} MB)")
print(f"Entrenado con PyTorch {torch.__version__} y torchvision {torchvision.__version__}: "
      f"el servidor debe usar versiones compatibles (se guardaron dentro del modelo).")


def cargar_modelo_para_inferencia(ruta):
    """Así se carga el modelo en el servidor: sin GPU y sin descargar pesos de internet."""
    contenido = torch.load(ruta, map_location="cpu", weights_only=False)
    if "bn_eps" not in contenido["config"]:
        raise RuntimeError("Este modelo no trae bn_eps (es de una versión anterior del cuaderno): "
                           "daría resultados distintos a los del entrenamiento. Vuelve a exportarlo con la versión 0.2.1 o posterior.")
    m = construir_modelo(preentrenado=False, bn_eps=contenido["config"]["bn_eps"])
    assert not any(isinstance(c, torch.nn.BatchNorm2d) for c in m.modules()), "quedó una normalización sin congelar"
    resultado = m.load_state_dict(contenido["state_dict"], strict=False)
    # Las capas congeladas no guardan num_batches_tracked; cualquier otra diferencia sí es un error
    sobrantes = [k for k in resultado.missing_keys if not k.endswith("num_batches_tracked")]
    if sobrantes or resultado.unexpected_keys:
        raise RuntimeError(f"El modelo no coincide: faltan {sobrantes}, sobran {resultado.unexpected_keys}")
    config = contenido["config"]
    for nombre, actual in (("torch", torch.__version__), ("torchvision", torchvision.__version__)):
        if config.get(nombre, "").split(".")[:2] != actual.split(".")[:2]:
            print(f"AVISO: el modelo se entrenó con {nombre} {config.get(nombre)} y aquí hay {actual}. "
                  f"Suele funcionar, pero conviene usar la misma versión.")
    return m.eval(), config


modelo_cpu, config_leida = cargar_modelo_para_inferencia(ruta_modelo)
zona_prueba = next(z for z in zonas if z["uso"] == "val")
x0, y0, x1, y1 = zona_prueba["caja"]
trozo = np.load(zona_prueba["imagen"]["cache"], mmap_mode="r")[y0:min(y1, y0 + LADO), x0:min(x1, x0 + LADO)]

cpu = torch.device("cpu")
umbral_exportado = config_leida["umbral_deteccion"]
t0 = time.time()
en_cpu = [d for d in detectar(modelo_cpu, trozo, cpu) if d["puntaje"] >= umbral_exportado]
segundos = time.time() - t0
modelo.to(cpu)
en_memoria = [d for d in detectar(modelo, trozo, cpu) if d["puntaje"] >= umbral_exportado]

# Comprobación directa: las características internas de los dos modelos deben ser prácticamente idénticas
def _a_tensor(t):
    t = np.asarray(t)[:LADO, :LADO]
    t = np.repeat(t[None], 3, axis=0) if t.ndim == 2 else t.transpose(2, 0, 1)
    return torch.from_numpy(np.ascontiguousarray(t)).float().div(255)


with torch.no_grad():
    caract_exportado = modelo_cpu.backbone(modelo_cpu.transform([_a_tensor(trozo)])[0].tensors)
    caract_original = modelo.backbone(modelo.transform([_a_tensor(trozo)])[0].tensors)
diferencia = max(float((caract_exportado[k] - caract_original[k]).abs().max()) for k in caract_original)
print(f"Diferencia máxima entre el modelo exportado y el original: {diferencia:.2e} (debe ser menor que 1e-3)")
assert diferencia < 1e-3, "El modelo exportado no calcula lo mismo que el original"
modelo.to(dispositivo)
print(f"Modelo exportado: {len(en_cpu)} detecciones; modelo en memoria: {len(en_memoria)} (deben ser casi iguales)")
# Pueden diferir en una o dos detecciones al borde del umbral por redondeos; más que eso es un error
assert abs(len(en_cpu) - len(en_memoria)) <= max(2, 0.05 * len(en_memoria)), \
    "El modelo exportado no da lo mismo que el original"

paso = LADO - SOLAPE
teselas_3584 = len(_posiciones(3584, LADO, paso)) * len(_posiciones(2748, LADO, paso))
print(f"En la CPU de esta máquina: {segundos:.1f} s por tesela de {LADO}x{LADO} px "
      f"-> una foto de 3584x2748 usa {teselas_3584} teselas, unos {segundos * teselas_3584 / 60:.1f} min.")
print("(La CPU gratuita de Colab y la del servidor del cliente pueden ser más lentas o más rápidas; es una referencia.)")

## Siguientes pasos
1. Revisa `curvas.png` y las fotos de `revision/`. Con 2 fotos es normal que la validación sea inestable.
2. Para mejorar: **etiqueta más fotos en CVAT** (de preferencia con aspecto variado: densas, escasas, con fibras) y
   exporta otra vez en COCO. Súbelas a `datos/imagenes/`, sustituye el JSON y corre de nuevo con otro `NOMBRE_CORRIDA`.
3. Cuando se decida qué es "cabeza" y qué es "cola" (pregunta pendiente al cliente), se puede cambiar el criterio de
   etiquetado y reentrenar.
4. Para usar el modelo en el sitio, `modelo_cabezas.pt` se copia al servidor y `detectar_cabezas` de
   `app/analisis/medicion.py` se reemplaza por la carga del modelo. Eso es un paso aparte: aquí no se toca el sitio.

## Historial de versiones
- **0.4.0**
  - Admite las fotos de campo claro, cuyos espermatozoides etiquetados miden hasta casi 700 px (en fluorescencia, hasta
    116): recortes y teselas de 1024 px (`LADO`), traslape de 768 (`SOLAPE`), anclas de 20 a 576 px y máscaras con
    más detalle (`ROI_MASCARA` = 28, máscaras de 56 x 56).
  - `INVERTIR_SI_FONDO_CLARO` = True: las fotos de campo claro se invierten para parecerse a las de fluorescencia
    (hipótesis sin comprobar).
  - Lote de 2 con tasa de aprendizaje 0.0025; recortes por época automáticos: 8 por foto, mínimo 200.
  - `SEPARACION` se queda en 100 px: las zonas de validación de fluorescencia son las mismas que en las corridas
    anteriores, para poder compararlas. La validación de campo claro tiene muy pocos espermatozoides.
  - Al final mide por separado fluorescencia y campo claro, para ver si sumar un tipo de foto afectó al otro.
  - **No sirve con los modelos de 0.3.x**: hay que entrenar de nuevo (cambian las anclas, las teselas y las máscaras).
- **0.3.1**
  - Solo cambia el número de versión; no hay cambios de código respecto a 0.3.0.
- **0.3.0**
  - Nueva limpieza al final de `detectar()`: quita los contornos que están (en `UMBRAL_CONTENIDO` = 50% o más) dentro de otro
    con más seguridad. En la prueba con las dos fotos bajó los contornos de más de 45 a 15, a costa de perder 4 aciertos.
    No hay que volver a entrenar: los modelos de 0.2.x siguen siendo válidos. El valor queda guardado en el modelo exportado
    (`umbral_contenido`) para que el servidor aplique la misma limpieza.
- **0.2.1**
  - Corrige un error de 0.2.0 al exportar: la comprobación fallaba con `batch_norm eps must be positive, but got 0.0`
    en las versiones recientes de PyTorch. Ahora el modelo cargado usa las mismas capas de normalización congeladas
    del entrenamiento. Los archivos `modelo_cabezas.pt` y `mejor.pt` de 0.2.0 siguen siendo válidos.
  - Cambiar de versión del cuaderno ya no impide reanudar una corrida.
- **0.2.0**
  - `MODO_PRUEBA` viene en `False` y la prueba guarda en su propia carpeta (`..._prueba`), sin mezclarse con la corrida real.
  - Al reanudar, compara la configuración guardada con la actual y se detiene si cambió.
  - Las fotos sin ninguna etiqueta se omiten (opción `INCLUIR_FOTOS_SIN_ETIQUETAS`).
  - `ultimo.pt` se guarda cada 10 épocas para no llenar Drive.
  - Al final prueba varios umbrales de seguridad y guarda el mejor dentro del modelo.
  - **Corrección importante:** el modelo exportado en 0.1.0 no calculaba lo mismo que el entrenado (las capas de
    normalización congeladas usan otro `eps` al cargarlo sin los pesos de COCO). Ahora se guarda `bn_eps` y se
    comprueba al exportar que ambos modelos dan los mismos resultados. **Los modelos exportados con 0.1.0 no sirven:
    hay que volver a entrenar o exportar.**
  - El modelo exportado guarda las versiones de PyTorch y torchvision y avisa si no coinciden al cargarlo.
  - `RECORTES_POR_EPOCA` automático según el número de fotos.
  - `detectar()` recibe el dispositivo como parámetro.
- **0.1.0**: primera versión (Mask R-CNN, recortes de 512 px, validación por zonas, reanudación y exportación a CPU).